# Hello!

This worksheet is designed as learning material for an undergraduate student with a strong understanding of calculus and some knowledge of orbital mechanics, but no prior experience with computational physics. It covers the basics of taking definite integrals and shows you how to start coding your own orbital dynamics simulations.

In [ ]:
# This cell handles the importation of the software libraries we'll need to run
# our simulations and visualize the resulting data.

!pip install -U plotly
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider


*Let's start with the basics.*

The essential equation in orbital dynamics is Newton's Law of Gravitation.

$F = \frac{m_1 \times m_2}{r^2}G$

And of course, we can construct the equation for the acceleration experienced by a body under the influence of this force my dividing by its mass. I'll add in directionality here with some vectors.

$\vec{a}_{m_1} = \frac{\vec{F}}{m_1} = -\frac{m_2}{r^2}G*\hat{r}$

Here, $\hat{r}$ points from $m_1$ towards $m_2$, but it would be useful to instead express $\vec{r}$ from an inertial reference frame. Say we define the centre of mass of the system as $\vec{R} = \frac{m_1 r_1 + m_2 r_2}{m_1 + m_2}$ It can be shown that $\frac{d^2\vec{R}}{dt^2} = 0$, so the barycenter acts as an inertial reference frame, and positions defined from this point of reference will allow us to take advantage of some physical constants. We can rearrange this into the general equation of motion for $m_1$.

$\frac{d^2 \vec{r}}{dt^2} + \frac{G(m_1+m_2)}{r^3}\vec{r} = 0$

The extra $r$ in the denominator of the second term comes from us expressing the direction as the non-unit vector $\vec{r}$ rather than $\hat{r}$. You might recognize this as an analytically solvable ODE (hint: it's separable, and angular momentum $L = \mu{}\vec{r}\times{}\frac{d\vec{r}}{dt}$ where $\mu{} = \frac{m_1 m_2}{m_1 + m_2}$ is constant). Depending on the energy of the system, the solution to this problem will trace a conic section.

Generally, the equation of motion is of the form $r(\theta{}) = \frac{\frac{L^2}{\mu{} k}}{1+ecos(\theta{}-\theta{}_0)}$, where $e$ and $\theta{}_0$ provide the degrees of freedom required to match the shape of the solution to any initial conditions, and $m_1$, $m_2$, and $L$ will scale the solution. The cell below provides a visual representation of what the solution looks like depending on the parameters of the problem, feel free to play around with the parameters to see how they effect the paths of the bodies. 

In [ ]:
G = 1.0

def orbit(e=0.3, theta0=0.0, L=1.0, m1=1.0, m2=1.0, angle=0.0):
    mu = (m1 * m2) / (m1 + m2)
    k = G * (m1 + m2)
    p = L**2 / (mu * k)

    if e > 1:
        theta_max = np.arccos(-1 / e)
        theta = np.linspace(theta0 - theta_max, theta0 + theta_max, 1000)
    else:
        theta = np.linspace(0, 2*np.pi, 1000)
    
    thetac = theta0 + angle

    r = p / (1 + e * np.cos(theta - theta0))
    r[r < 0] = np.nan

    x = r * np.cos(theta)
    y = r * np.sin(theta)

    rc = p / (1 + e * np.cos(thetac - theta0))
    if rc < 0:
        rc = np.nan

    xc = rc * np.cos(thetac)
    yc = rc * np.sin(thetac)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='Orbit'))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Barycentre'))
    fig.add_trace(go.Scatter(x=[xc], y=[yc], mode='markers', name='m1', marker=dict(size=20)))

    rmax = max(r)

    fig.update_xaxes(range=[-rmax*1.2, rmax*1.2])
    fig.update_yaxes(range=[-rmax*1.2, rmax*1.2])

    fig.update_layout(
        width=600,
        height=600,
        title=f"Orbit (e={e:.2f})"
    )

    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

interact(
    orbit,
    e=FloatSlider(min=0.0, max=2.0, step=0.01, value=0.3),
    theta0=FloatSlider(min=0.0, max=2*np.pi, step=0.01, value=0.0),
    L=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0),
    m1=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
    m2=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
    angle=FloatSlider(min=0.0, max=2*np.pi, step=0.01, value=0.0)
)

You might notice that only one of the bodies has a traced path in this plot. *This is because we can solve for the paths of the bodies (but not the positions w.r.t. time) independently.* We have reduced the two body problem to two *reduced* two body problems; $m_1$ and the barycentre, and $m_2$ and the barycentre. These solutions are called reduced because one of the bodies, the barycentre, is considered to be immobile. Note that the speed at which the body traverses its orbit is not consistent with the theta parameter you are able to play with here, as expressing the position of the body in its orbit in closed solution of the one body problem involves elliptic functions, the application of which is beyond the scope of this worksheet. I hope you'll accept this simpler parametrization instead.

We've now seen the closed form solution of the case of two bodies interacting gravitationally. What happens, though, if we wish to examine the motion of three or more bodies? Unfortunately, this is where our luck runs out. There has yet to be a solution discovered for the case of three or more bodies interacting gravitationally. Instead, we'll have to look for a way to approximate our orbits.

Remember the equation we used earlier for the acceleration of a body undergoing gravitational attraction to another body:

$\vec{a}_{m_1} = -\frac{m_2}{r^2}G * \hat{r}$

If we generalize this equation to *n* bodies, we have

$\vec{a}_{m_i} = \sum^n_{1, i\neq j}{\frac{m_j}{r_{ij}^2}G * \hat{r}_{ij}}$

Let's move to cartesian co-ordinates to avoid having to deal with polar terms in our equations of motion. We'll again use the barycentre of the system as our inertial reference frame. In a given co-ordinate $X_a$, the motion of a single body is described by the following equation:

$\ddot{X_a} = \sum^n_{1, i\neq j}{\frac{m_im_j}{r_{ij}^2}G * (\hat{r}_{ij}\cdotp{}\hat{X_a})}$

Applying this to the two-body system we're already familiar with, we get

$\ddot{X}_{m1} = \frac{m_2G(X_{m2}-X_{m1})}{r_{12}^2}$, 
$\ddot{Y}_{m1} = \frac{m_2G(Y_{m2}-Y_{m1})}{r_{12}^2}$, 
$\ddot{Z}_{m1} = \frac{m_2G(Z_{m2}-Z_{m1})}{r_{12}^2}$, 
$\ddot{X}_{m2} = \frac{m_1G(X_{m1}-X_{m2})}{r_{21}^2}$, 
$\ddot{Y}_{m2} = \frac{m_1G(Y_{m1}-Y_{m2})}{r_{21}^2}$, 
$\ddot{Z}_{m2} = \frac{m_1G(Z_{m1}-Z_{m2})}{r_{21}^2}$,

We now have 6 2nd order ODEs. Generally, numerical integrators are designed to work with 1st order differential equations. How do we get around this? Simple! We can define, for some co-ordinate $X_a$, the first order derivative as some new variable, and substitute. We make the substitution $\dot{X}_a = U_a$, and $\ddot{X}_a = \dot{U}_a$. 